In this notebook we will train a `Word2Vec` model.

We will use the `gensim` library which offers extremely fast training on the CPU.

We will again rely on `polars` and its small memory footprint to load and process the data. To speed things up, let's use the dataset in a parquet format (we won't have to deal with `jasonl` files anymore). [I shared the dataset here](https://www.kaggle.com/datasets/radek1/otto-full-optimized-memory-footprint). 

Why are we training word2vec embeddings in the first place?

A session where one action follows another action is very much like... a sentence! In sentences, words that are related appear together. We don't necessarily expect to see the word 'spaceship' in a sentence discussing various ways to cook a steak. The word "steak" is more likely to appear close to words such as rosemary, salt, pepper, oil and butter. In this sense, these words are thematically related. And with a large enough corpus we can start making further distinctions! Maybe butter will appear closer in the embedding space to milk than for instance to orange juice, even though both are drinks you can have with your breakfast! (that might be due to milk having the property of being a substance used to produce butter, which might tip the embeddings for "milk" and "butter" closer together assuming our corpus would contain texts on butter production!).

Similarly here we can exploit the fact that `aids` appearing in a sequence close together likely share some similarity. A person browsing for gardening equipment is probably not looking at surfboards and vice versa.

Once we train our model, what will we be able to use it for? First and foremost, candidate generation! Though one might also imagine using it for scoring. Essentially, a model such as this can be very handy in the context of session-based recommendation models!

Let's get to work! 🙂

## Other resources you might find useful:

* [💡 [2 methods] How-to ensemble predictions 🏅🏅🏅](https://www.kaggle.com/code/radek1/2-methods-how-to-ensemble-predictions)
* [co-visitation matrix - simplified, imprvd logic 🔥](https://www.kaggle.com/code/radek1/co-visitation-matrix-simplified-imprvd-logic)
* [💡 Word2Vec How-to [training and submission]🚀🚀🚀](https://www.kaggle.com/code/radek1/word2vec-how-to-training-and-submission)
* [local validation tracks public LB perfecty -- here is the setup](https://www.kaggle.com/competitions/otto-recommender-system/discussion/364991)
* [💡 For my friends from Twitter and LinkedIn -- here is how to dive into this competition 🐳](https://www.kaggle.com/competitions/otto-recommender-system/discussion/368560)
* [Full dataset processed to CSV/parquet files with optimized memory footprint](https://www.kaggle.com/competitions/otto-recommender-system/discussion/363843)

# Data Preprocessing

In [ ]:
!pip install polars

import polars as pl
from gensim.test.utils import common_texts
from gensim.models import Word2Vec

train = pl.read_parquet('../input/otto-full-optimized-memory-footprint/train.parquet')
test = pl.read_parquet('../input/otto-full-optimized-memory-footprint/test.parquet')

Let us now transform the data into a format that the `gensim` library can work with. Thanks to `polars` we can do so very efficiently and very quickly.

There are various ways we could feed our data to our model, however doing so straight from RAM in the form of Python lists is probably one of the fastest! As we have enough resources on Kaggle to do so, let us take this approach!

In [ ]:
sentences_df = pl.concat([train, test]).groupby('session').agg(
    pl.col('aid').alias('sentence')
)

In [ ]:
sentences = sentences_df['sentence'].to_list()

Time to train our model.

# Training a word2vec model

In [ ]:
%%time

w2vec = Word2Vec(sentences=sentences, vector_size=32, min_count=1, workers=4)

With the model fully train, let us use similarity between trained representations of our `aids` to create a submission.

The search functionality where we look for nearest neighbors in the embedding space is built into `gensim`, but it is unfortunately super slow. Let's use `annoy` which is much faster (it performs approximate nearest neigbor search).

In [ ]:
%%time

from annoy import AnnoyIndex

aid2idx = {aid: i for i, aid in enumerate(w2vec.wv.index_to_key)}
index = AnnoyIndex(32, 'euclidean')

for aid, idx in aid2idx.items():
    index.add_item(idx, w2vec.wv.vectors[idx])
    
index.build(10)

Let's create a submission! 🙂

# Outputting a submission

In [ ]:
import pandas as pd
import numpy as np

from collections import defaultdict

sample_sub = pd.read_csv('../input/otto-recommender-system//sample_submission.csv')

session_types = ['clicks', 'carts', 'orders']
test_session_AIDs = test.to_pandas().reset_index(drop=True).groupby('session')['aid'].apply(list)
test_session_types = test.to_pandas().reset_index(drop=True).groupby('session')['type'].apply(list)

labels = []

type_weight_multipliers = {0: 1, 1: 6, 2: 3}
for AIDs, types in zip(test_session_AIDs, test_session_types):
    if len(AIDs) >= 20:
        # if we have enough aids (over equals 20) we don't need to look for candidates! we just use the old logic
        weights=np.logspace(0.1,1,len(AIDs),base=2, endpoint=True)-1
        aids_temp=defaultdict(lambda: 0)
        for aid,w,t in zip(AIDs,weights,types): 
            aids_temp[aid]+= w * type_weight_multipliers[t]
            
        sorted_aids=[k for k, v in sorted(aids_temp.items(), key=lambda item: -item[1])]
        labels.append(sorted_aids[:20])
    else:
        # here we don't have 20 aids to output -- we will use word2vec embeddings to generate candidates!
        AIDs = list(dict.fromkeys(AIDs[::-1]))
        
        # let's grab the most recent aid
        most_recent_aid = AIDs[0]
        
        # and look for some neighbors!
        nns = [w2vec.wv.index_to_key[i] for i in index.get_nns_by_item(aid2idx[most_recent_aid], 21)[1:]]
                        
        labels.append((AIDs+nns)[:20])

Let's now pull it all together and write to a file,

In [ ]:
labels_as_strings = [' '.join([str(l) for l in lls]) for lls in labels]

predictions = pd.DataFrame(data={'session_type': test_session_AIDs.index, 'labels': labels_as_strings})

prediction_dfs = []

for st in session_types:
    modified_predictions = predictions.copy()
    modified_predictions.session_type = modified_predictions.session_type.astype('str') + f'_{st}'
    prediction_dfs.append(modified_predictions)

submission = pd.concat(prediction_dfs).reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

And we are done!

Thank you for reading! Happy Kaggling! 🙌

# BONUS: How to use word2vec to generate candidates/features for training a 2-stage recommender

Just like a covisiation matrix, for any `AID` `word2vec` can give us a list of `AIDs` resembling our query `AID`. The output will be ordered starting with `AIDs` that are most alike.

In order for us to visualize what is happening, let me give you a simplified example.

## Mock data

In our data we have `aids` organized by `sessions`.

In [ ]:
data = pl.DataFrame(data={'session': [0, 0, 1, 1], 'aid': [10, 20, 20, 30], 'type': [0, 0, 1, 0]})
data

We can use word2vec to generate candidates. For instance, maybe using word2vec we would generate the following candidates for the sessions in our data:

`{0: [11, 20], 1: [25, 6]}`

We can reshape our candidates to look as follows:

In [ ]:
candidates = pl.DataFrame(data={'session': [0, 0, 1, 1], 'aid': [11, 20, 25, 6]})
candidates

As you see, for our canddiates we don't have too much information apart from `session` and `aid`! This is exactly like the out put `word2vec` can give us!

And that is okay. Our ranker can deal with that. For some rows we will have information in this or that column, for another we won't. This is not an issue to a ranking model, `NAs` or `nulls` (depending on the framework) also can carry signal (absence of some type of data).

Here, our ranker will see that we don't have `type` information for candidates... but we will create another important column that will allow it to uniquely identify our candidates as coming from `word2vec`.

Here are the key steps.

### 1. Add ordering information to our candidates.

The order is important! A candidate appearing earlier in the list of candidates in some sense has a higher score, is more similar to the AIDs in a session (remember, `word2vec` can give us output ordered by similarity in descending order).

In [ ]:
candidates = candidates.with_column(pl.col('aid').cumcount().over('session').alias('word2vec_rank') + 1)
candidates

### 2. Merge this information onto candidates.

Now, we need to take this information and add this onto our original data.

But how do we add candidates?! If we just concat these dataframes together, we will have a duplicate entry for session `0` for `aid` of 20.

What we need to do is a join but of a specific kind! (hello SQL ideas 👋).

We want to keep the rows that are already there in `data`, append information to them where there is a match AND create new rows if there isn't.

`outer join` to the rescue!

In [ ]:
data = data.join(candidates, on=['session', 'aid'], how='outer')
data

Beautiful! We added candidates, we didn't duplicate rows, information got appended to wehre it should go!

Plus, we added a reflection of the ordering coming from the `word2vec` model in the form of the `word2vec_rank` column.

And the code to do it all is provided above!

### 3. How to practice the above / put it to good use to improve your LB standing.

1. Generate candidates using word2vec from this notebook.
2. Go to [🏆 Training an XGBoost Ranker on the GPU 🔥🔥🔥](https://www.kaggle.com/code/radek1/training-an-xgboost-ranker-on-the-gpu) and add the features to the training and test data.
3. Rerun training and see your score improve.

Hope this can be of help! 🙂 If you found this useful, please upvote the notebook! Thank you 🙏

Also, if you have any questions, let me know. I'll do my best to try to address them (this is how this bonus section came about, it answers the questions several people kept asking me! 🙂)

Thank you for reading!